# Fashion-MNIST s_c_star_mean by epoch (hard5)

Train and test curves for the hard5 group from Fashion-MNIST audit JSONL files.

In [1]:
from pathlib import Path
import json

import matplotlib.pyplot as plt

In [2]:
def load_series(path):
    train = {}
    test = {}
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            record = json.loads(line)
            if record.get("group") != "hard5":
                continue
            if "s_c_star_mean" not in record:
                continue
            epoch = record.get("epoch")
            split = record.get("split")
            value = record.get("s_c_star_mean")
            if split == "train":
                train[int(epoch)] = float(value)
            elif split == "test":
                test[int(epoch)] = float(value)
    return train, test


def plot_train_test(dnll_path, logistic_path, out_dir):
    dnll_train, dnll_test = load_series(dnll_path)
    log_train, log_test = load_series(logistic_path)

    if not (dnll_train or dnll_test or log_train or log_test):
        raise ValueError("No hard5 s_c_star_mean data found in inputs")

    epochs = sorted(set(dnll_train) | set(dnll_test) | set(log_train) | set(log_test))

    def series(vals, epochs):
        return [vals.get(e) for e in epochs]

    fig, axes = plt.subplots(1, 2, figsize=(9.2, 4.2), constrained_layout=True, sharey=True)

    title_size = 14
    label_size = 13
    tick_size = 12
    legend_size = 12

    train_ax = axes[0]
    if dnll_train:
        train_ax.plot(epochs, series(dnll_train, epochs), marker="o", label="DNLL Loss")
    if log_train:
        train_ax.plot(epochs, series(log_train, epochs), marker="s", label="Logistic Loss")
    train_ax.set_title("Fashion-MNIST Train", fontsize=title_size)
    train_ax.set_xlabel("epoch", fontsize=label_size)
    train_ax.set_ylabel(r"avg $s_{c^*}$ (top 5% hardest)", fontsize=label_size)
    train_ax.grid(True, linestyle="--", alpha=0.4)
    train_ax.tick_params(axis="both", labelsize=tick_size)
    train_ax.legend(fontsize=legend_size)

    test_ax = axes[1]
    if dnll_test:
        test_ax.plot(epochs, series(dnll_test, epochs), marker="o", label="DNLL Loss")
    if log_test:
        test_ax.plot(epochs, series(log_test, epochs), marker="s", label="Logistic Loss")
    test_ax.set_title("Fashion-MNIST Test", fontsize=title_size)
    test_ax.set_xlabel("epoch", fontsize=label_size)
    test_ax.grid(True, linestyle="--", alpha=0.4)
    test_ax.tick_params(axis="both", labelsize=tick_size)
    test_ax.legend(fontsize=legend_size)

    out_path = out_dir / "fashionmnist_s_c_star_mean_hard5_train_test.png"
    fig.savefig(out_path, dpi=600)
    plt.close(fig)
    return out_path

In [3]:
results_dir = Path("..") / "results"
paths = sorted(results_dir.glob("audit_fashionmnist_*_seed1234.jsonl"))

dnll_path = next((p for p in paths if "dnll" in p.stem), None)
logistic_path = next((p for p in paths if "logistic" in p.stem), None)

if dnll_path is None or logistic_path is None:
    raise FileNotFoundError("Expected Fashion-MNIST dnll and logistic audit files in results_dir")

out_dir = Path(".")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = plot_train_test(dnll_path, logistic_path, out_dir)
print(out_path)

fashionmnist_s_c_star_mean_hard5_train_test.png
